# M2SVid tiny Colab demo

Smallest useful demo: first view the repo's precomputed demo outputs, then optionally run M2SVid refinement on the precomputed warped video/mask.

Recommended GPU: **L4**. A100 is easiest if available. T4 may OOM.

## 0. Mount Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
!python --version
!df -h /content /content/drive

## 1. Clone repo and set paths

In [ ]:
from pathlib import Path
import time, subprocess, shutil, os
DRIVE_ROOT = Path('/content/drive/MyDrive/m2svid_demo')
CKPT_DIR = DRIVE_ROOT / 'ckpts'
OUTPUT_ARCHIVE = DRIVE_ROOT / 'outputs'
for p in [DRIVE_ROOT, CKPT_DIR, OUTPUT_ARCHIVE]: p.mkdir(parents=True, exist_ok=True)
%cd /content
if not Path('/content/m2svid').exists():
    !git clone --depth 1 https://github.com/google-research/m2svid.git /content/m2svid
%cd /content/m2svid
!find demo -maxdepth 3 -type f | sort

## 2. Quick sanity check: view precomputed outputs shipped with the repo

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path
def show_video(path, width=720):
    path=Path(path); assert path.exists(), f'Missing {path}'
    data='data:video/mp4;base64,'+b64encode(path.read_bytes()).decode()
    display(HTML(f'<video width="{width}" controls loop><source src="{data}" type="video/mp4"></video><p><code>{path}</code></p>'))
print('Input'); show_video('/content/m2svid/demo/input.mp4', 520)
print('Warped right'); show_video('/content/m2svid/demo/reprojected/input_reprojected.mp4', 520)
print('Mask'); show_video('/content/m2svid/demo/reprojected/input_reprojected_mask.mp4', 520)
print('M2SVid side-by-side'); show_video('/content/m2svid/demo/m2svid/input_sbs.mp4', 720)

## 3. Download checkpoints

M2SVid weight is public. Hi3D `ckpts.zip` is on Google Drive; this uses `gdown`. If gdown fails, manually place `ckpts.zip` or `open_clip_pytorch_model.bin` under `MyDrive/m2svid_demo/ckpts/`.

In [ ]:
%cd /content/m2svid
!pip -q install gdown
if not (CKPT_DIR/'m2svid_weights.pt').exists():
    !wget -c -O {CKPT_DIR/'m2svid_weights.pt'} https://storage.googleapis.com/gresearch/m2svid/m2svid_weights.pt
else:
    print('M2SVid weight already in Drive')
hi3d_zip = CKPT_DIR/'hi3d_ckpts.zip'
if not hi3d_zip.exists() and not (CKPT_DIR/'open_clip_pytorch_model.bin').exists():
    !gdown --fuzzy 'https://drive.google.com/file/d/1j_NEG2CPhFeRetYziWK6Qe62R5h7lG_V/view?usp=sharing' -O {hi3d_zip}
!mkdir -p /content/m2svid/ckpts
if hi3d_zip.exists() and not Path('/content/m2svid/ckpts/open_clip_pytorch_model.bin').exists():
    !unzip -n {hi3d_zip} -d /content/m2svid/
if (CKPT_DIR/'open_clip_pytorch_model.bin').exists():
    !ln -sf {CKPT_DIR/'open_clip_pytorch_model.bin'} /content/m2svid/ckpts/open_clip_pytorch_model.bin
!ln -sf {CKPT_DIR/'m2svid_weights.pt'} /content/m2svid/ckpts/m2svid_weights.pt
!find ckpts -maxdepth 2 -type f | sort | head -50

## 4. Install minimal dependencies for refinement-only run

In [ ]:
%cd /content/m2svid
!pip -q install omegaconf einops pytorch-lightning==1.5.0 kornia==0.6.9 open-clip-torch==2.29.0 pytorch-msssim==1.0.0 ffmpeg-python av imageio safetensors xformers || true
!python - <<'PY'
import torch, torchvision
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('torchvision', torchvision.__version__)
PY

## 5. Actual tiny inference

Use precomputed `demo/reprojected/*` and run only `inpaint_and_refine.py`. This avoids DepthCrafter setup for the first trial.

In [ ]:
%cd /content/m2svid
import shutil, time, subprocess
out = Path('outputs/m2svid_tiny')
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True, exist_ok=True)
cmd = PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${PYTHONPATH}" python inpaint_and_refine.py \
  --mask_antialias 0 \
  --model_config configs/m2svid.yaml \
  --ckpt ckpts/m2svid_weights.pt \
  --video_path demo/input.mp4 \
  --reprojected_path demo/reprojected/input_reprojected.mp4 \
  --reprojected_mask_path demo/reprojected/input_reprojected_mask.mp4 \
  --output_folder outputs/m2svid_tiny
start=time.time(); print(cmd)
ret=subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed:', round(time.time()-start,1),'sec')
!find outputs/m2svid_tiny -type f -print
!nvidia-smi

## 6. View and save output

In [ ]:
out_dir=Path('/content/m2svid/outputs/m2svid_tiny')
for p in sorted(out_dir.glob('*.mp4')): print(p, round(p.stat().st_size/1e6,2),'MB')
if (out_dir/'input_sbs.mp4').exists(): show_video(out_dir/'input_sbs.mp4', 720)
elif (out_dir/'input_generated.mp4').exists(): show_video(out_dir/'input_generated.mp4', 520)
archive = OUTPUT_ARCHIVE / time.strftime('tiny_demo_%Y%m%d_%H%M%S')
archive.mkdir(parents=True, exist_ok=True)
for p in out_dir.glob('*'): shutil.copy2(p, archive/p.name)
print('Saved to', archive)

## 7. Optional: create a few examples by varying stereo strength

This reuses the shipped `demo/depthcrafter/input.npz`, so no DepthCrafter setup is needed.

In [ ]:
%cd /content/m2svid
DISPARITIES=[0.03,0.05]
for d in DISPARITIES:
    tag=f'disp_{str(d).replace(".","p")}'
    print('===', tag, '===')
    !mkdir -p outputs/reprojected_{tag} outputs/m2svid_{tag}
    !PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${PYTHONPATH}" python warping.py --video_path demo/input.mp4 --depth_path demo/depthcrafter/input.npz --output_path_reprojected outputs/reprojected_{tag}/input_reprojected.mp4 --output_path_mask outputs/reprojected_{tag}/input_reprojected_mask.mp4 --disparity_perc {d}
    !PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${PYTHONPATH}" python inpaint_and_refine.py --mask_antialias 0 --model_config configs/m2svid.yaml --ckpt ckpts/m2svid_weights.pt --video_path demo/input.mp4 --reprojected_path outputs/reprojected_{tag}/input_reprojected.mp4 --reprojected_mask_path outputs/reprojected_{tag}/input_reprojected_mask.mp4 --output_folder outputs/m2svid_{tag}
!find outputs -maxdepth 2 -path '*m2svid_disp*' -type f -print

## Notes for custom videos

After this works, upload a short MP4 to Drive, resize to dimensions divisible by 64, run DepthCrafter to make a `.npz`, then run `warping.py` and `inpaint_and_refine.py`. For the first trial, use the shipped demo.